# Local Features Lab

This notebook reorganizes the `#5 지역 특징` lecture into a compact practice set.

Topics:
- Harris corner response
- Harris corners on a real image
- SIFT keypoints and descriptors
- FLANN-based feature matching

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = Path("lab-session/04-local-features")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

VIEW_PATH = DATA_DIR / "view.jpg"
BUS_PATH = DATA_DIR / "bus.jpg"
BUS2_PATH = DATA_DIR / "bus2.jpg"
print("Base dir:", BASE_DIR.resolve())

## Harris response on a toy pattern

In [ ]:
toy = np.array(
    [
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=np.float32,
)
ux = np.array([[-1, 0, 1]], dtype=np.float32)
uy = ux.T
g = np.outer(cv2.getGaussianKernel(3, 1), cv2.getGaussianKernel(3, 1))

dx = cv2.filter2D(toy, cv2.CV_32F, ux)
dy = cv2.filter2D(toy, cv2.CV_32F, uy)
dxx = dx * dx
dyy = dy * dy
dxy = dx * dy

sxx = cv2.filter2D(dxx, cv2.CV_32F, g)
syy = cv2.filter2D(dyy, cv2.CV_32F, g)
sxy = cv2.filter2D(dxy, cv2.CV_32F, g)
harris = (sxx * syy - sxy * sxy) - 0.04 * (sxx + syy) ** 2

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(toy, cmap="gray")
axes[0].set_title("Toy pattern")
axes[1].imshow(harris, cmap="jet")
axes[1].set_title("Harris response")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Harris corners on a real image

In [ ]:
view_bgr = cv2.imread(str(VIEW_PATH))
if view_bgr is None:
    raise FileNotFoundError(VIEW_PATH)
view_gray = cv2.cvtColor(view_bgr, cv2.COLOR_BGR2GRAY)
response = cv2.cornerHarris(np.float32(view_gray), blockSize=2, ksize=3, k=0.04)
response = cv2.dilate(response, None)
threshold = 0.01 * response.max()
corner_overlay = view_bgr.copy()
corner_overlay[response > threshold] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(view_gray, cmap="gray")
axes[0].set_title("Gray")
axes[1].imshow(response, cmap="jet")
axes[1].set_title("Corner response")
axes[2].imshow(cv2.cvtColor(corner_overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title("Detected corners")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
print("Threshold:", threshold)

## SIFT keypoints and descriptors

In [ ]:
if not hasattr(cv2, "SIFT_create"):
    raise RuntimeError("This OpenCV build does not support SIFT.")

sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(view_gray, None)
sift_preview = cv2.drawKeypoints(
    view_gray,
    keypoints,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)

plt.figure(figsize=(7, 5))
plt.imshow(sift_preview, cmap="gray")
plt.title(f"SIFT keypoints: {len(keypoints)}")
plt.axis("off")
print("Descriptor shape:", descriptors.shape)
print(descriptors[:2])

## FLANN-based matching

In [ ]:
bus1 = cv2.imread(str(BUS_PATH))
bus2 = cv2.imread(str(BUS2_PATH))
if bus1 is None or bus2 is None:
    raise FileNotFoundError("bus matching images are missing")

bus1_crop = bus1[10:668, 114:944]
gray1 = cv2.cvtColor(bus1_crop, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(bus2, cv2.COLOR_BGR2GRAY)
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

matcher = cv2.DescriptorMatcher_create(cv2.DescriptorMatcher_FLANNBASED)
knn = matcher.knnMatch(des1, des2, 2)
good = []
for first, second in knn:
    if (first.distance / second.distance) < 0.7:
        good.append(first)

match_view = cv2.drawMatches(
    bus1_crop,
    kp1,
    bus2,
    kp2,
    good[:60],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

plt.figure(figsize=(16, 6))
plt.imshow(cv2.cvtColor(match_view, cv2.COLOR_BGR2RGB))
plt.title(f"Good matches: {len(good)}")
plt.axis("off")

## Applied summary

This creates a compact output image that can be reused as a lab summary.

In [ ]:
summary_top = np.hstack([
    cv2.resize(cv2.cvtColor(view_bgr, cv2.COLOR_BGR2RGB), (320, 240)),
    cv2.resize(cv2.cvtColor(corner_overlay, cv2.COLOR_BGR2RGB), (320, 240)),
])
summary_bottom = np.hstack([
    cv2.resize(cv2.cvtColor(sift_preview, cv2.COLOR_BGR2RGB), (320, 240)),
    cv2.resize(cv2.cvtColor(match_view, cv2.COLOR_BGR2RGB), (320, 240)),
])
summary = np.vstack([summary_top, summary_bottom])
summary_path = OUTPUT_DIR / "local_features_summary.jpg"
cv2.imwrite(str(summary_path), cv2.cvtColor(summary, cv2.COLOR_RGB2BGR))

plt.figure(figsize=(10, 7))
plt.imshow(summary)
plt.title("Local features summary")
plt.axis("off")
print("Saved summary to:", summary_path.resolve())

## What to run next

Run the OpenCV window-based demos from this folder:
- `python examples/01_harris_response_toy.py`
- `python examples/02_harris_corners_real.py`
- `python examples/03_sift_keypoints.py`
- `python examples/04_sift_flann_match.py`